[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Constraints


## What you will be able to do

Declare the rules a table enforces, `NOT NULL`, `UNIQUE`, `CHECK`, `PRIMARY KEY` and foreign keys,
and read the `IntegrityError` each one raises. Switch foreign keys on for every connection, outside
any transaction, and find the rows that broke them while they were off. Answer a duplicate with an
upsert instead of an error, and know what `INSERT OR IGNORE` and `INSERT OR REPLACE` do besides.
Catch a constraint error by its name, and avoid the two tests that `NULL` quietly fails.


## The idea

### The problem

Every table in this guide so far has accepted whatever it was given. Load the same export twice and
every reading appears twice, so every count and mean after that is wrong. Insert a reading for
station 99, which does not exist, and the reading sits in the table where no join will ever find it.
Type 93.0 for -9.3 and the Arctic winter gains a temperature no thermometer there has ever read. A
program can check for all of that before every insert, but then every program that writes to the
file has to remember to, and check it the same way.

A constraint is a rule that the table keeps for itself, so the check happens whichever program
writes. The **Tables and Queries** notebook already declared one that was never enforced:
`REFERENCES stations (id)`, a foreign key that SQLite ignores unless the connection asks it not to.
And once a table refuses duplicates, a program that loads the same reading twice, or corrects one,
needs a way to say what should happen instead of an error.

### What a constraint is

> A **constraint** is a rule declared in `CREATE TABLE` that SQLite checks on every insert and
> update, refusing a change that breaks it with `sqlite3.IntegrityError`. `NOT NULL` refuses a
> missing value, **`UNIQUE`** refuses a value, or a combination of values, that another row already
> holds, `CHECK` refuses a row for which an expression is false, `PRIMARY KEY` names the column that
> identifies a row, and a **foreign key**, declared with `REFERENCES`, refuses a value that is not
> in the table it names, and says with `ON DELETE` what happens to the rows that point at a row
> being deleted. An **upsert** is an `INSERT` with an `ON CONFLICT` clause, which says what to do
> when the new row would break a uniqueness constraint: nothing, or an update of the row that is
> already there.

### Why it works that way

- **A constraint lives with the data.** Every program that writes to the file meets the same rule, in
  any language, and so does a person typing into the sqlite3 shell.
- **Foreign keys are off until a connection asks.** SQLite's documentation says they are disabled by
  default, for backwards compatibility, and must be enabled separately for every connection, with
  `PRAGMA foreign_keys = ON`. The setting is not stored in the file.
- **That PRAGMA does nothing inside a transaction.** SQLite's documentation says enabling or
  disabling foreign keys in the middle of a transaction does not return an error, and simply has no
  effect. Under `autocommit=False`, which the **autocommit and isolation_level** notebook
  recommends, a transaction is always open.
- **`UNIQUE` builds an index.** SQLite enforces a uniqueness constraint with an index it creates for
  the purpose, which keeps the check fast and speeds up queries on those columns too.
- **`NULL` equals nothing, not even another `NULL`.** So `UNIQUE` lets any number of rows hold
  `NULL`, and a comparison such as `!=` with `NULL` is neither true nor false, so `WHERE` leaves the
  row out.
- **The error name is for code, and the message is for people.** `sqlite_errorname` says which kind
  of constraint failed, such as `'SQLITE_CONSTRAINT_UNIQUE'`. A message is written to be read, and
  its wording can change between versions of SQLite, as `no such column` did in SQLite 3.46.

### Where this shows up

Every relational database enforces these same kinds of constraint. PostgreSQL, in the **asyncpg and
psycopg3, Deep Dive** guide, enforces foreign keys without being asked, and SQLite's documentation
says its upsert follows the syntax PostgreSQL established, with generalizations. The **SQLAlchemy,
Deep Dive** guide declares constraints on its models, and turns them into the SQL this notebook
writes by hand. A validation library such as the one in the **Pydantic, Deep Dive** guide checks
values in Python before they reach any database, which catches a mistake earlier, but only in the
program that uses it.

### What this notebook covers

- Declaring `NOT NULL`, `UNIQUE`, `CHECK`, `PRIMARY KEY` and `REFERENCES`
- What each constraint refuses, and the message it gives
- Foreign keys, switched on for every connection, and `PRAGMA foreign_key_check`
- `ON DELETE`: the default, and `CASCADE`
- Upserts with `ON CONFLICT`, `excluded` and `RETURNING`
- What `INSERT OR IGNORE` and `INSERT OR REPLACE` do besides
- Catching a constraint error by its name, and where `IntegrityError` sits among sqlite3's errors
- When to use `DO NOTHING`, `DO UPDATE`, `OR IGNORE` or `OR REPLACE`
- A loader that can run twice, and applies corrections when it runs again
- Seven errors: a load run twice, an `ON CONFLICT` naming the wrong columns, foreign keys switched
  on inside a transaction, `INSERT OR IGNORE` dropping a bad reading, `INSERT OR REPLACE` deleting
  notes, `UNIQUE` beside `NULL`, and `!=` beside `NULL`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE);
    CREATE TABLE readings (station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL,
                           celsius REAL CHECK (celsius BETWEEN -90 AND 60),
                           UNIQUE (station_id, hour));
    INSERT INTO stations (name) VALUES ('Svalbard');
""")

insert = "INSERT INTO readings VALUES (?, ?, ?)"
rows = [(1, "2025-03-01T06:00", -13.1), (1, "2025-03-01T06:00", -13.1),
        (1, "2025-03-01T07:00", 93.0), (2, "2025-03-01T06:00", -1.0)]
for row in rows:
    try:
        conn.execute(insert, row)
        print(row, "stored")
    except sqlite3.IntegrityError as error:
        print(row, "refused:", error)

upsert = insert + " ON CONFLICT (station_id, hour) DO UPDATE SET celsius = excluded.celsius"
conn.execute(upsert, (1, "2025-03-01T06:00", -13.4))
print(conn.execute("SELECT * FROM readings").fetchall())
conn.close()
```

```
(1, '2025-03-01T06:00', -13.1) stored
(1, '2025-03-01T06:00', -13.1) refused: UNIQUE constraint failed: readings.station_id, readings.hour
(1, '2025-03-01T07:00', 93.0) refused: CHECK constraint failed: celsius BETWEEN -90 AND 60
(2, '2025-03-01T06:00', -1.0) refused: FOREIGN KEY constraint failed
[(1, '2025-03-01T06:00', -13.4)]
```

Four readings, and three refusals. The table already had Svalbard's reading for 06:00, it would not
believe 93.0 degrees, and there is no station 2. The last line did not insert the duplicate or raise
an error either: the upsert found the reading already there, and updated its temperature instead.


## Setup

Five imports, the year of readings, and each station's code, the three letters of its nearest
airport. The worked examples create the tables, since their constraints are what this notebook is
about.

- `sqlite3` creates the tables and runs every statement
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
CODES = {"Bergen": "BGO", "Oslo": "OSL", "Svalbard": "LYR", "Tromso": "TOS", "Kirkenes": "KKN"}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


print("ready to build", DATABASE)


ready to build scratch/stations.db


## Worked examples

### Declaring constraints

The stations and their readings, as the **Tables and Queries** notebook designed them, now with the
rules they always implied, and a third table for notes about the stations. `name` is `UNIQUE`, and
`code` is `UNIQUE` too. A reading's `station_id` refers to a station, its temperature has to be
plausible, and no station has two readings for one hour. A note refers to its station with
`ON DELETE CASCADE`. `CONSTRAINT plausible_celsius` gives a `CHECK` a name, which its error message
will use:


In [2]:
conn = sqlite3.connect(DATABASE, autocommit=False)
conn.executescript("""
    CREATE TABLE stations (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL UNIQUE,
        code     TEXT UNIQUE,
        latitude REAL NOT NULL CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90)
    ) STRICT;

    CREATE TABLE readings (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        source     TEXT,
        UNIQUE (station_id, hour)
    ) STRICT;

    CREATE TABLE notes (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id) ON DELETE CASCADE,
        note       TEXT NOT NULL
    ) STRICT;
""")

station_ids = {name: conn.execute("INSERT INTO stations (name, code, latitude) VALUES (?, ?, ?)",
                                  (name, CODES[name], latitude)).lastrowid
               for name, latitude in LATITUDES.items()}
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 ((station_ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
conn.executemany("INSERT INTO notes (station_id, note) VALUES (?, ?)",
                 [(station_ids["Kirkenes"], "Heater checked before winter"), (station_ids["Kirkenes"], "Mast repainted")])
conn.commit()

for name, kind in conn.execute("SELECT name, type FROM sqlite_schema ORDER BY type DESC, name"):
    print(f"{kind:<6} {name}")


table  notes
table  readings
table  stations
index  sqlite_autoindex_readings_1
index  sqlite_autoindex_stations_1
index  sqlite_autoindex_stations_2


Three tables, and three indexes nobody wrote: SQLite created one for every `UNIQUE` constraint, named
`sqlite_autoindex_` and the table, so a new station's name or a new reading's hour is checked against
an index instead of the whole table. The tables are `STRICT`, as the **Type Affinity** notebook
recommends, which is a constraint on types of its own.

### What each constraint refuses

Five changes, each breaking one rule, run one at a time. `IntegrityError` says which rule, and where:


In [3]:
INSERT_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"
attempts = [
    ("a reading with no hour", INSERT_READING, (station_ids["Oslo"], None, -3.0)),
    ("a second reading for one hour", INSERT_READING, (station_ids["Oslo"], "2025-01-01T00:00", -3.0)),
    ("a second station called Oslo", "INSERT INTO stations (name, code, latitude) VALUES (?, ?, ?)", ("Oslo", None, 59.9)),
    ("a reading of 93.0 degrees", INSERT_READING, (station_ids["Oslo"], "2026-01-01T00:00", 93.0)),
    ("a reading for station 99", INSERT_READING, (99, "2026-01-01T00:00", -3.0)),
]

for description, sql, parameters in attempts:
    try:
        conn.execute(sql, parameters)
        print(f"{description:<30} accepted")
    except sqlite3.IntegrityError as error:
        print(f"{description:<30} refused: {error}")
conn.commit()


a reading with no hour         refused: NOT NULL constraint failed: readings.hour
a second reading for one hour  refused: UNIQUE constraint failed: readings.station_id, readings.hour
a second station called Oslo   refused: UNIQUE constraint failed: stations.name
a reading of 93.0 degrees      refused: CHECK constraint failed: plausible_celsius
a reading for station 99       accepted


Four refusals name what they refused: the column that was `NULL`, the columns whose values already
existed, and the `CHECK` by the name it was given. The fifth change was accepted, and committed. A
reading for a station that does not exist is now in the file, because the foreign key was declared
and never switched on.

### Foreign keys, switched on for every connection

`PRAGMA foreign_keys` reports whether a connection enforces foreign keys, and it starts at 0 on every
connection. Setting it has an effect only while no transaction is open, which, under
`autocommit=False`, is never. `open_database` opens the connection with `autocommit=True`, where
nothing is open, sets the `PRAGMA`, and then switches to `autocommit=False`.
`PRAGMA foreign_key_check` finds the rows that broke a foreign key while it was off:


In [4]:
print("foreign keys on this connection:", conn.execute("PRAGMA foreign_keys").fetchone()[0])
conn.close()


def open_database(path):
    """A connection that enforces foreign keys, with autocommit=False, as this guide recommends."""
    conn = sqlite3.connect(path, autocommit=True)       # no transaction is open yet, so the PRAGMA takes effect
    conn.execute("PRAGMA foreign_keys = ON")
    conn.autocommit = False
    return conn


conn = open_database(DATABASE)
print("foreign keys on the new connection:", conn.execute("PRAGMA foreign_keys").fetchone()[0])
print("rows that break a foreign key:", conn.execute("PRAGMA foreign_key_check").fetchall())

conn.execute("DELETE FROM readings WHERE station_id NOT IN (SELECT id FROM stations)")
conn.commit()
try:
    conn.execute(INSERT_READING, (99, "2026-01-01T00:00", -3.0))
except sqlite3.IntegrityError as error:
    print("a reading for station 99, now:", error)


foreign keys on this connection: 0
foreign keys on the new connection: 1
rows that break a foreign key: [('readings', 35041, 'stations', 0)]
a reading for station 99, now: FOREIGN KEY constraint failed


`foreign_key_check` returned the table, the row's `rowid`, the table the key refers to, and which of
the table's foreign keys it broke, so the reading for station 99 could be found and deleted. With the
`PRAGMA` on, the same insert is refused. The message names no table or column, so
`foreign_key_check` is where to look for the row that caused it.

### ON DELETE: the default, and CASCADE

A foreign key also decides what happens when the row it points at is deleted. The default,
`NO ACTION`, refuses to leave rows pointing at nothing, while `ON DELETE CASCADE` deletes them along
with the row they point at. `notes` cascades, and `readings` does not:


In [5]:
def count_notes(conn, station):
    """How many notes a station has."""
    return conn.execute("SELECT COUNT(*) FROM notes WHERE station_id = ?", (station_ids[station],)).fetchone()[0]


print("Kirkenes notes before:", count_notes(conn, "Kirkenes"))
conn.execute("DELETE FROM stations WHERE name = ?", ("Kirkenes",))
print("Kirkenes notes after deleting the station:", count_notes(conn, "Kirkenes"))

try:
    conn.execute("DELETE FROM stations WHERE name = ?", ("Bergen",))
except sqlite3.IntegrityError as error:
    print("deleting Bergen:", error)

conn.rollback()
print("after rollback, Kirkenes notes:", count_notes(conn, "Kirkenes"))


Kirkenes notes before: 2
Kirkenes notes after deleting the station: 0
deleting Bergen: FOREIGN KEY constraint failed
after rollback, Kirkenes notes: 2


Deleting Kirkenes took its two notes with it, and deleting Bergen was refused, since 8,760 readings
point at it. The rollback put Kirkenes back. `CASCADE` suits rows that mean nothing without their
parent, such as notes, and the default suits rows worth more than the parent's deletion, such as a
year of readings. SQLite's documentation recommends an index on the columns that refer to another
table, since every delete of a station has to look them up, which the **Indexes and Query Plans**
notebook covers.

### Upserts: ON CONFLICT

`ON CONFLICT`, followed by the columns of a uniqueness constraint, says what to do when an insert
would break that constraint. `DO NOTHING` skips the row. `DO UPDATE` changes the row already there,
and in its `SET`, `excluded` names the values the insert tried to add. `RETURNING` hands back
columns from the row that was inserted or updated:


In [6]:
SKIP_READING = """
    INSERT INTO readings (station_id, hour, celsius, source) VALUES (?, ?, ?, ?)
    ON CONFLICT (station_id, hour) DO NOTHING
    RETURNING id
"""
UPSERT_READING = """
    INSERT INTO readings (station_id, hour, celsius, source) VALUES (?, ?, ?, ?)
    ON CONFLICT (station_id, hour) DO UPDATE SET celsius = excluded.celsius, source = excluded.source
    RETURNING id, celsius, source
"""
first_oslo = (station_ids["Oslo"], "2025-01-01T00:00")
reading = "SELECT id, celsius, source FROM readings WHERE station_id = ? AND hour = ?"

print("before:                ", conn.execute(reading, first_oslo).fetchone())
print("DO NOTHING returned:   ", conn.execute(SKIP_READING, (*first_oslo, -3.4, "correction")).fetchall())
print("DO UPDATE returned:    ", conn.execute(UPSERT_READING, (*first_oslo, -3.4, "correction")).fetchall())
new_hour = (station_ids["Oslo"], "2026-01-01T00:00", -2.9, "export")
print("a new hour, inserted:  ", conn.execute(UPSERT_READING, new_hour).fetchall())
conn.commit()


before:                 (2, -3.5, None)
DO NOTHING returned:    []
DO UPDATE returned:     [(2, -3.4, 'correction')]
a new hour, inserted:   [(35041, -2.9, 'export')]


`DO NOTHING` left the reading alone and returned no row. `DO UPDATE` changed the same row, id 2, and
`RETURNING` said so. That is how to learn which row an upsert touched, since `lastrowid` reports the
last row inserted, and an update inserts none. For an hour with no reading, the same statement simply
inserted one.

### What INSERT OR IGNORE and INSERT OR REPLACE do besides

Two older forms look like upserts. `INSERT OR IGNORE` skips a row that breaks a constraint, and
`INSERT OR REPLACE` deletes the rows in the way and inserts the new one:


In [7]:
cursor = conn.execute("INSERT OR IGNORE INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)", (*first_oslo, -3.3))
print("OR IGNORE changed", cursor.rowcount, "rows:", conn.execute(reading, first_oslo).fetchone())

conn.execute("INSERT OR REPLACE INTO readings (station_id, hour, celsius, source) VALUES (?, ?, ?, ?)",
             (*first_oslo, -3.3, "replacement"))
print("OR REPLACE:             ", conn.execute(reading, first_oslo).fetchone())
conn.commit()


OR IGNORE changed 0 rows: (2, -3.4, 'correction')
OR REPLACE:              (35042, -3.3, 'replacement')


`OR IGNORE` skipped the duplicate, as `DO NOTHING` did, but it skips a row that breaks `NOT NULL` or
a `CHECK` just as quietly, which one of the Common errors shows. `OR REPLACE` did replace the
reading, but as a new row with a new id. That is harmless for a reading nothing points at, and not
for a station, whose deletion cascades to its notes, which another of the Common errors shows.

### Catching a constraint error by its name

`IntegrityError` is one branch of sqlite3's exceptions: every one of them is a `sqlite3.Error`, and
`except sqlite3.Error` catches them all. Since Python 3.11, an error that SQLite raised carries
`sqlite_errorname` and `sqlite_errorcode`, which say exactly which constraint failed, so code can
answer a duplicate differently from an implausible value, and re-raise anything else:


In [8]:
print(" -> ".join(cls.__name__ for cls in sqlite3.IntegrityError.__mro__[:4]))

ANSWERS = {"SQLITE_CONSTRAINT_UNIQUE": "already there", "SQLITE_CONSTRAINT_CHECK": "implausible"}


def add_reading(conn, station, hour, celsius):
    """Add one reading, and say why not when a known constraint refuses it."""
    try:
        with conn:
            conn.execute(INSERT_READING, (station_ids[station], hour, celsius))
        return "added"
    except sqlite3.IntegrityError as error:
        if error.sqlite_errorname in ANSWERS:
            return f"refused, {ANSWERS[error.sqlite_errorname]} ({error.sqlite_errorname}, code {error.sqlite_errorcode})"
        raise


for hour, celsius in [("2026-01-02T00:00", -3.1), ("2026-01-02T00:00", -3.1), ("2026-01-02T01:00", 93.0)]:
    print(hour, celsius, "->", add_reading(conn, "Oslo", hour, celsius))


IntegrityError -> DatabaseError -> Error -> Exception
2026-01-02T00:00 -3.1 -> added
2026-01-02T00:00 -3.1 -> refused, already there (SQLITE_CONSTRAINT_UNIQUE, code 2067)
2026-01-02T01:00 93.0 -> refused, implausible (SQLITE_CONSTRAINT_CHECK, code 275)


The first reading was added, the second found its hour taken, and the third failed its `CHECK`, each
told apart by name rather than by the words of the message. A foreign key or `NOT NULL` failure would
have been re-raised, since `add_reading` has no answer for it.

### DO NOTHING, DO UPDATE, OR IGNORE or OR REPLACE

Four ways to insert a row that may already exist:

| Write | When | Why |
|---|---|---|
| `ON CONFLICT (columns) DO NOTHING` | a row that may already be there and should stay as it is, such as a load run again | it skips only a row that breaks the named uniqueness constraint, and raises for anything else |
| `ON CONFLICT (columns) DO UPDATE SET ... = excluded....` | a row whose newer values should win, such as a corrected reading | it changes the existing row in place, keeping its id and every row that points at it |
| `INSERT OR IGNORE` | rarely: a row that may break any constraint and may be dropped without a word | it skips a row that fails `NOT NULL` or a `CHECK` as quietly as a duplicate |
| `INSERT OR REPLACE` | rarely: a row nothing points at, whose id does not matter | it deletes the old row and inserts a new one, with a new id, and a cascade takes rows that pointed at the old one |

The default is `ON CONFLICT`, naming the constraint, with `DO NOTHING` or `DO UPDATE`.

### A loader that can run twice

The pieces of this notebook in one job: an export of stations and readings is loaded so that running
it again changes nothing, and running a corrected export updates what changed. Stations are upserted
by name and return their ids, readings are upserted by station and hour, and the whole export is one
transaction, so an export with an implausible reading loads nothing at all:


In [9]:
def load_export(conn, export):
    """Load an export's stations and readings: a second run changes nothing, and corrections win."""
    with conn:
        ids = {}
        for name, code, latitude in export["stations"]:
            (ids[name],) = conn.execute("""
                INSERT INTO stations (name, code, latitude) VALUES (?, ?, ?)
                ON CONFLICT (name) DO UPDATE SET code = excluded.code, latitude = excluded.latitude
                RETURNING id
            """, (name, code, latitude)).fetchone()
        for name, hour, celsius in export["readings"]:
            conn.execute("""
                INSERT INTO readings (station_id, hour, celsius, source) VALUES (?, ?, ?, 'export')
                ON CONFLICT (station_id, hour) DO UPDATE SET celsius = excluded.celsius, source = excluded.source
            """, (ids[name], hour, celsius))
    return ids


def totals(conn):
    """The number of stations, and of readings on 3 January 2026."""
    stations = conn.execute("SELECT COUNT(*) FROM stations").fetchone()[0]
    readings = conn.execute("SELECT COUNT(*) FROM readings WHERE hour LIKE '2026-01-03%'").fetchone()[0]
    conn.commit()
    return stations, readings


export = {
    "stations": [("Kirkenes", "KKN", 69.73), ("Bjornoya", None, 74.50)],
    "readings": [("Bjornoya", "2026-01-03T00:00", -6.2), ("Bjornoya", "2026-01-03T01:00", -6.4)],
}
print("first run: ", load_export(conn, export), "totals:", totals(conn))
print("second run:", load_export(conn, export), "totals:", totals(conn))

export["readings"][1] = ("Bjornoya", "2026-01-03T01:00", -6.5)
ids = load_export(conn, export)
print("a corrected reading:", conn.execute(reading, (ids["Bjornoya"], "2026-01-03T01:00")).fetchone())

export["readings"].append(("Bjornoya", "2026-01-03T02:00", 66.0))
try:
    load_export(conn, export)
except sqlite3.IntegrityError as error:
    print("an implausible export:", error.sqlite_errorname, "| totals:", totals(conn))


first run:  {'Kirkenes': 5, 'Bjornoya': 6} totals: (6, 2)
second run: {'Kirkenes': 5, 'Bjornoya': 6} totals: (6, 2)
a corrected reading: (35045, -6.5, 'export')
an implausible export: SQLITE_CONSTRAINT_CHECK | totals: (6, 2)


The second run returned the same ids and left both totals as they were, since every row it tried to
add was already there. The corrected export changed one reading in place. The export with a reading
of 66.0 degrees broke a `CHECK`, which no `ON CONFLICT` answers, and the `with` block rolled back
the whole export, so nothing from that run was loaded.

### Where each part came from

| In the loader | What it relies on | The section that showed it |
|---|---|---|
| `ON CONFLICT (name) DO UPDATE ... RETURNING id` | an upsert that keeps a station's id, and says what it is | Upserts: ON CONFLICT |
| `excluded.celsius` | the value the insert tried to add | Upserts: ON CONFLICT |
| `UNIQUE (station_id, hour)` | the constraint the readings' upsert names | Declaring constraints |
| the export with 66.0 refused | a `CHECK`, which no upsert answers | What each constraint refuses |
| `error.sqlite_errorname` | an error told apart by name, not by message | Catching a constraint error by its name |
| `open_database` for `conn` | foreign keys enforced on every connection | Foreign keys, switched on for every connection |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/12-constraints-solutions.ipynb).

**1.** Try to add a second station called `Oslo`, and print the exception's class name, its
`sqlite_errorname` and its `sqlite_errorcode`.


In [10]:
# your code here


**2.** Add Bergen's reading for `2025-06-21T12:00` again with `ON CONFLICT DO NOTHING`, and show that
the reading already stored is unchanged.


In [11]:
# your code here


**3.** Correct Tromso's reading for `2025-01-01T00:00` to -2.4 with an upsert that returns the row's
id, and show that the id is the one the reading had before.


In [12]:
# your code here


**4.** Open a second connection with foreign keys enforced, print its `PRAGMA foreign_keys`, and show
that it refuses a note for station 99.


In [13]:
# your code here


**5.** Try to delete Svalbard, print the error, and count the rows in each table that point at
Svalbard.


In [14]:
# your code here


**6.** Count the stations whose `code` is not `'OSL'`, first with `!=` and then with `IS NOT`, and
explain why the two counts differ.


In [15]:
# your code here


## Common errors

### sqlite3.IntegrityError: UNIQUE constraint failed: readings.station_id, readings.hour


In [16]:
night = [(station_ids["Oslo"], "2026-01-04T00:00", -4.0), (station_ids["Oslo"], "2026-01-04T01:00", -4.2)]
conn.executemany(INSERT_READING, night)
conn.commit()                                              # the export, loaded

conn.executemany(INSERT_READING, night)                    # the same export, loaded again


IntegrityError: UNIQUE constraint failed: readings.station_id, readings.hour

The export was loaded once, and running it again met a reading already stored for the first row, so
`executemany` stopped there. Had the duplicate come later in the list, the rows before it would have
been inserted and left pending. A load that may run twice says what a duplicate means, with an
upsert, so a second run changes nothing:


In [17]:
conn.rollback()
cursor = conn.executemany(INSERT_READING + " ON CONFLICT (station_id, hour) DO NOTHING", night)
conn.commit()

print("rows added by the second run:", cursor.rowcount)


rows added by the second run: 0


### sqlite3.OperationalError: ON CONFLICT clause does not match any PRIMARY KEY or UNIQUE constraint


In [18]:
conn.execute(INSERT_READING + " ON CONFLICT (hour) DO UPDATE SET celsius = excluded.celsius",
             (station_ids["Oslo"], "2026-01-04T00:00", -4.1))


OperationalError: ON CONFLICT clause does not match any PRIMARY KEY or UNIQUE constraint

An upsert's conflict target has to name the columns of a uniqueness constraint exactly, and no
constraint is on `hour` alone: many stations share an hour. The constraint is on the station and the
hour together, so the target names both:


In [19]:
conn.execute(INSERT_READING + " ON CONFLICT (station_id, hour) DO UPDATE SET celsius = excluded.celsius",
             (station_ids["Oslo"], "2026-01-04T00:00", -4.1))
print(conn.execute(reading, (station_ids["Oslo"], "2026-01-04T00:00")).fetchone())
conn.commit()


(35046, -4.1, None)


### No error, and a reading for a station that does not exist: foreign keys switched on inside a transaction


In [20]:
hasty = sqlite3.connect(DATABASE, autocommit=False)
hasty.execute("PRAGMA foreign_keys = ON")
hasty.execute(INSERT_READING, (99, "2026-01-05T00:00", -5.0))
hasty.commit()

print("foreign keys on hasty:", hasty.execute("PRAGMA foreign_keys").fetchone()[0])
hasty.close()


foreign keys on hasty: 0


The connection asked for foreign keys, and the insert of a reading for station 99 went through. Under
`autocommit=False` a transaction was already open when the `PRAGMA` ran, and SQLite ignores that
`PRAGMA` inside a transaction, without an error, so `PRAGMA foreign_keys` still reports 0. Set it
before any transaction opens, as `open_database` does, and look for the rows that got in:


In [21]:
print("rows that break a foreign key:", conn.execute("PRAGMA foreign_key_check").fetchall())
conn.execute("DELETE FROM readings WHERE station_id NOT IN (SELECT id FROM stations)")
conn.commit()

careful = open_database(DATABASE)
print("foreign keys on careful:", careful.execute("PRAGMA foreign_keys").fetchone()[0])
try:
    careful.execute(INSERT_READING, (99, "2026-01-05T00:00", -5.0))
except sqlite3.IntegrityError as error:
    print("the same insert:", error)
careful.close()


rows that break a foreign key: [('readings', 35048, 'stations', 0)]
foreign keys on careful: 1
the same insert: FOREIGN KEY constraint failed


### No error, and a reading dropped without a word: INSERT OR IGNORE


In [22]:
batch = [(station_ids["Oslo"], "2026-01-06T00:00", -6.1),
         (station_ids["Oslo"], "2026-01-06T01:00", 61.0),
         (station_ids["Oslo"], None, -6.3)]
cursor = conn.executemany("INSERT OR IGNORE INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)", batch)
conn.commit()

print("rows added:", cursor.rowcount, "of", len(batch))


rows added: 1 of 3


`OR IGNORE` was meant to skip duplicates, and there were none. It skipped a reading of 61.0 degrees,
which failed the `CHECK`, and a reading with no hour, which failed `NOT NULL`, and said nothing
about either, so two readings that deserved a look were simply lost. `ON CONFLICT ... DO NOTHING`
skips only what breaks the constraint it names, and raises for everything else:


In [23]:
try:
    with conn:
        conn.executemany(INSERT_READING + " ON CONFLICT (station_id, hour) DO NOTHING", batch)
except sqlite3.IntegrityError as error:
    print("refused:", error)


refused: CHECK constraint failed: plausible_celsius


### No error, and a station's notes gone: INSERT OR REPLACE


In [24]:
kirkenes = "SELECT id, latitude FROM stations WHERE name = 'Kirkenes'"
print("before:", conn.execute(kirkenes).fetchone(), "notes:", count_notes(conn, "Kirkenes"))

conn.execute("INSERT OR REPLACE INTO stations (name, code, latitude) VALUES (?, ?, ?)", ("Kirkenes", "KKN", 69.72))
(new_id, latitude) = conn.execute(kirkenes).fetchone()
notes_left = conn.execute("SELECT COUNT(*) FROM notes WHERE station_id IN (?, ?)",
                          (station_ids["Kirkenes"], new_id)).fetchone()[0]
print("after: ", (new_id, latitude), "notes:", notes_left)


before: (5, 69.73) notes: 2
after:  (7, 69.72) notes: 0


The change was meant to correct Kirkenes's latitude by a hundredth of a degree. `OR REPLACE` deleted
the Kirkenes row, whose name was in the way, and inserted a new one with a new id, and the delete
cascaded to both of its notes. Nothing raised. Against Bergen, whose readings do not cascade, the
same statement would have failed on the foreign key instead. Roll it back before it is committed,
and correct a row in place with `DO UPDATE`:


In [25]:
conn.rollback()
conn.execute("""
    INSERT INTO stations (name, code, latitude) VALUES (?, ?, ?)
    ON CONFLICT (name) DO UPDATE SET latitude = excluded.latitude
""", ("Kirkenes", "KKN", 69.72))
conn.commit()

print("after DO UPDATE:", conn.execute(kirkenes).fetchone(), "notes:", count_notes(conn, "Kirkenes"))


after DO UPDATE: (5, 69.72) notes: 2


### No error, and two stations with no code: UNIQUE beside NULL


In [26]:
conn.execute("INSERT INTO stations (name, code, latitude) VALUES (?, ?, ?)", ("Jan Mayen", None, 70.94))
conn.commit()

print(conn.execute("SELECT name, code FROM stations WHERE code IS NULL ORDER BY name").fetchall())


[('Bjornoya', None), ('Jan Mayen', None)]


`code` is `UNIQUE`, and two stations have none. For `UNIQUE`, every `NULL` is different from every
other, so any number of rows can hold one. That is right if a code is optional, and wrong if every
station must have one, which takes `NOT NULL` as well. Adding it to an existing table needs the
rebuild the **Changing a Schema** notebook shows, so here it is on a new table:


In [27]:
conn.execute("CREATE TABLE coded_stations (name TEXT NOT NULL UNIQUE, code TEXT NOT NULL UNIQUE) STRICT")
try:
    conn.execute("INSERT INTO coded_stations VALUES (?, ?)", ("Jan Mayen", None))
except sqlite3.IntegrityError as error:
    print("refused:", error)
conn.rollback()


refused: NOT NULL constraint failed: coded_stations.code


### No error, and thousands of readings left out: != beside NULL


In [28]:
not_exported = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND source != 'export'",
                            (station_ids["Oslo"],)).fetchone()[0]
print("Oslo readings not from an export:", not_exported)
conn.commit()


Oslo readings not from an export: 1


Almost every one of Oslo's readings has no `source` at all, and the count found one. `source !=
'export'` is not true for a `NULL` source, nor false, and `WHERE` keeps only the rows for which it is
true, so every reading with no source was left out. `IS NOT` treats `NULL` as a value like any other,
and `IS NULL` asks for it directly:


In [29]:
not_exported = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND source IS NOT 'export'",
                            (station_ids["Oslo"],)).fetchone()[0]
with_no_source = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND source IS NULL",
                              (station_ids["Oslo"],)).fetchone()[0]
print("Oslo readings not from an export:", not_exported, "| with no source at all:", with_no_source)
conn.close()


Oslo readings not from an export: 8764 | with no source at all: 8763


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [30]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `NOT NULL`, `UNIQUE`, `CHECK`, `PRIMARY KEY` and `REFERENCES` make a table refuse bad rows itself,
  with `sqlite3.IntegrityError`, and naming a constraint names it in the message.
- Foreign keys are enforced only on a connection that ran `PRAGMA foreign_keys = ON` outside a
  transaction, and `PRAGMA foreign_key_check` finds the rows that got in while they were not.
- `ON DELETE CASCADE` deletes the rows that point at a deleted row, and the default refuses the
  delete.
- `ON CONFLICT (columns) DO NOTHING` or `DO UPDATE SET ... = excluded....` answers a duplicate, and
  `RETURNING` says which row an upsert touched.
- `INSERT OR IGNORE` also drops rows that fail `NOT NULL` or `CHECK`, and `INSERT OR REPLACE`
  deletes and reinserts, with a new id and a cascade.
- Tell constraint errors apart by `sqlite_errorname`, not by the words of the message.
- `UNIQUE` allows many `NULL`s, and `!=` leaves `NULL` rows out, so `IS NULL` and `IS NOT` are the
  tests for them.


## What is next

The **Changing a Schema** notebook changes tables that already hold data: adding a column, the
rebuild that other changes need, with foreign keys switched off before its transaction begins, and
`PRAGMA user_version` as a record of which changes a file has had.


---

&#8592; **Previous:** [autocommit and isolation_level](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/11-autocommit-and-isolation-level.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Changing a Schema](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/13-changing-a-schema.ipynb) &#8594;
